In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/unsw-bot-iot/bot_iot_merged.csv


In [1]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif, chi2

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef
)

from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier



In [2]:
DATA_PATH = "/kaggle/input/unsw-bot-iot"
CHUNK_SIZE = 200_000

csv_files = []
for root, _, files in os.walk(DATA_PATH):
    for f in files:
        if f.endswith(".csv"):
            csv_files.append(os.path.join(root, f))

chunks = []
for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print("Dataset shape:", df.shape)


/tmp/ipykernel_55/1563457183.py:12: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
/tmp/ipykernel_55/1563457183.py:12: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
/tmp/ipykernel_55/1563457183.py:12: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
/tmp/ipykernel_55/1563457183.py:12: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
/tmp/ipykernel_55/1563457183.py:12: DtypeWarning: Columns (7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file, chunksize=CHUNK_SIZE):
/tmp/ipykernel_55/1563457183.p

Dataset shape: (3668522, 46)


In [3]:
TARGET = "category"
drop_cols = ['pkSeqID', 'stime', 'ltime', 'saddr', 'daddr', 'attack', 'category', 'subcategory']
if TARGET in drop_cols:
    drop_cols.remove(TARGET)
df.drop(columns=drop_cols, inplace=True, errors='ignore')

In [4]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

print("Libraries loaded ")

Libraries loaded 


In [5]:
from sklearn.impute import SimpleImputer

# Convert ports to numeric
for col in ['sport','dport']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Replace -1 with NaN
df[['sport','dport']] = df[['sport','dport']].replace(-1, np.nan)

# Median imputation
imputer = SimpleImputer(strategy='median')
df[['sport','dport']] = imputer.fit_transform(df[['sport','dport']])

In [6]:
drop_cols = ['proto_number','flgs_number','state_number']
df.drop(columns=drop_cols, inplace=True, errors='ignore')

In [7]:
categorical_cols = ['proto', 'state', 'flgs']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df[TARGET] = le.fit_transform(df[TARGET])

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [10]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef
)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [12]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# Compute MI scores
mi_scores = mutual_info_classif(
    X_train,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "feature": X_train.columns,
    "mi_score": mi_scores
}).sort_values(by="mi_score", ascending=False)

print(mi_df.head(20))

               feature  mi_score
5                  dur  0.773556
25  AR_P_Proto_P_DstIP  0.766096
24  AR_P_Proto_P_SrcIP  0.747592
16               srate  0.736214
15                rate  0.727651
28  AR_P_Proto_P_Sport  0.726248
29  AR_P_Proto_P_Dport  0.726188
8                  sum  0.642816
6                 mean  0.625571
10                 max  0.617889
7               stddev  0.530885
19           TnBPDstIP  0.441250
26   N_IN_Conn_P_DstIP  0.402723
9                  min  0.357655
18           TnBPSrcIP  0.334177
13              sbytes  0.297533
3                bytes  0.284277
21          TnP_PDstIP  0.242080
23       TnP_Per_Dport  0.238817
22        TnP_PerProto  0.218889


In [12]:
X_train_chi = X_train.copy()
X_train_chi[X_train_chi < 0] = 0
from sklearn.feature_selection import chi2
import numpy as np

chi_scores, p_values = chi2(X_train_chi, y_train)

chi_df = pd.DataFrame({
    "feature": X_train.columns,
    "chi2_score": chi_scores,
    "p_value": p_values
}).sort_values(by="chi2_score", ascending=False)

print(chi_df.head(20))

               feature    chi2_score  p_value
3                bytes  1.779887e+12      0.0
14              dbytes  1.424507e+12      0.0
13              sbytes  9.670463e+11      0.0
18           TnBPSrcIP  8.590286e+11      0.0
19           TnBPDstIP  5.840217e+11      0.0
29  AR_P_Proto_P_Dport  6.091008e+10      0.0
28  AR_P_Proto_P_Sport  5.219436e+10      0.0
24  AR_P_Proto_P_SrcIP  3.802165e+10      0.0
25  AR_P_Proto_P_DstIP  3.244616e+10      0.0
15                rate  2.743701e+10      0.0
1                dport  1.711873e+10      0.0
4                  seq  7.328529e+09      0.0
16               srate  7.161894e+09      0.0
22        TnP_PerProto  4.874556e+09      0.0
12               dpkts  3.883212e+08      0.0
2                 pkts  2.459450e+08      0.0
11               spkts  1.308385e+08      0.0
23       TnP_Per_Dport  1.105542e+08      0.0
20          TnP_PSrcIP  1.062411e+08      0.0
21          TnP_PDstIP  8.419486e+07      0.0


In [12]:
import numpy as np

def fisher_score(X, y):
    scores = []
    classes = np.unique(y)

    for col in X.columns:
        num = 0.0
        den = 0.0
        overall_mean = X[col].mean()

        for c in classes:
            X_c = X[y == c][col]
            n_c = len(X_c)

            if n_c == 0:
                continue

            mean_c = X_c.mean()
            var_c = X_c.var()

            num += n_c * (mean_c - overall_mean) ** 2
            den += n_c * var_c

        scores.append(num / den if den != 0 else 0)

    return pd.Series(scores, index=X.columns)
# Compute Fisher scores
fisher_scores = fisher_score(X_train, y_train)

fisher_df = pd.DataFrame({
    "feature": fisher_scores.index,
    "fisher_score": fisher_scores.values
}).sort_values(by="fisher_score", ascending=False)

print(fisher_df.head(20))


               feature  fisher_score
22        TnP_PerProto      1.595005
36           state_CON      0.548757
1                dport      0.299938
26   N_IN_Conn_P_DstIP      0.232688
25  AR_P_Proto_P_DstIP      0.231586
5                  dur      0.106498
32          proto_icmp      0.105786
45           state_URP      0.104341
6                 mean      0.096574
10                 max      0.095423
18           TnBPSrcIP      0.080816
38           state_FIN      0.073652
20          TnP_PSrcIP      0.069574
8                  sum      0.069037
23       TnP_Per_Dport      0.065483
19           TnBPDstIP      0.065241
24  AR_P_Proto_P_SrcIP      0.065060
11               spkts      0.061920
21          TnP_PDstIP      0.059290
4                  seq      0.055676


In [15]:
# Rank features for each method
mi_rank = mi_df[['feature', 'mi_score']].copy()
mi_rank['mi_rank'] = mi_rank['mi_score'].rank(ascending=False, method='average')

chi_rank = chi_df[['feature', 'chi2_score']].copy()
chi_rank['chi_rank'] = chi_rank['chi2_score'].rank(ascending=False, method='average')

fisher_rank = fisher_df[['feature', 'fisher_score']].copy()
fisher_rank['fisher_rank'] = fisher_rank['fisher_score'].rank(ascending=False, method='average')


rank_df = mi_rank[['feature', 'mi_rank']] \
    .merge(chi_rank[['feature', 'chi_rank']], on='feature') \
    .merge(fisher_rank[['feature', 'fisher_rank']], on='feature')


rank_df['avg_rank'] = rank_df[['mi_rank', 'chi_rank', 'fisher_rank']].mean(axis=1)


rank_df = rank_df.sort_values('avg_rank').reset_index(drop=True)
rank_df
top_10_features = rank_df['feature'].head(10).tolist()
print("Top 10 selected features:", top_10_features)

Top 10 selected features: ['AR_P_Proto_P_DstIP', 'AR_P_Proto_P_SrcIP', 'TnBPSrcIP', 'dur', 'TnBPDstIP', 'TnP_PerProto', 'AR_P_Proto_P_Dport', 'AR_P_Proto_P_Sport', 'sbytes', 'rate']


In [ ]:
X_train_sel = X_train[top_10_features].copy()
X_test_sel  = X_test[top_10_features].copy()

In [14]:
# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

NameError: name 'X_train_sel' is not defined

In [13]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, class_weights))

sample_weights = np.array([class_weight_dict[y] for y in y_train])


In [14]:
results = []


In [15]:
def train_evaluate(
    model,
    model_name,
    X_train,
    y_train,
    X_test,
    y_test,
    sample_weight=None,
    experiment_config=None
):
    if experiment_config is None:
        experiment_config = {
            "experiment_id": "UNKNOWN",
            "experiment_type": "UNKNOWN",
            "feature_set": "UNKNOWN",
            "num_features": X_train.shape[1]
        }

    start_time = time.time()

    # Train (use sample_weight only if supported)
    try:
        if sample_weight is not None:
            model.fit(X_train, y_train, sample_weight=sample_weight)
        else:
            model.fit(X_train, y_train)
    except TypeError:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    elapsed = time.time() - start_time

    results.append({
        # Experiment metadata
        "Experiment_ID": experiment_config["experiment_id"],
        "Experiment_Type": experiment_config["experiment_type"],
        "Feature_Set": experiment_config["feature_set"],
        "Num_Features": experiment_config["num_features"],

        # Model
        "Model": model_name,

        # Metrics
        "Accuracy": accuracy_score(y_test, y_pred),

        "Macro_Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro_Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro_F1": f1_score(y_test, y_pred, average="macro", zero_division=0),

        "Weighted_Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Weighted_Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "Weighted_F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),

        "Balanced_Accuracy": balanced_accuracy_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),

        "Train_Time_sec": elapsed,

        # Diagnostics
        "Confusion_Matrix": confusion_matrix(y_test, y_pred),
        "Per_Class_F1": f1_score(y_test, y_pred, average=None, zero_division=0)
    })

    print(f"✅ {model_name} finished in {elapsed:.2f}s")


In [25]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp1_Top10_CombinedRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "Top10_MI_ChiSq_Fisher",
    "num_features": X_train_scaled.shape[1]
}


In [26]:
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)


✅ GaussianNB finished in 1.18s
✅ LDA finished in 2.87s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ LogisticRegression finished in 1591.39s
✅ LinearSVM finished in 3234.90s
✅ KNN finished in 43.47s
✅ DecisionTree finished in 38.51s
✅ RandomForest finished in 318.77s


In [27]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

display(results_df)


,Experiment_ID,Experiment_Type,Feature_Set,Num_Features,Model,Accuracy,Macro_Precision,Macro_Recall,Macro_F1,Weighted_Precision,Weighted_Recall,Weighted_F1,Balanced_Accuracy,MCC,Train_Time_sec,Confusion_Matrix,Per_Class_F1
0,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,DecisionTree,0.999982,0.997834,0.995742,0.996782,0.999982,0.999982,0.999982,0.995742,0.999966,38.508450,"[[385321, 4, 0, 0, 0], [1, 330049, 0, 2, 0], [...","[0.999990916746794, 0.999986365846749, 0.98412..."
1,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,RandomForest,0.999992,0.997828,0.970788,0.983469,0.999992,0.999992,0.999992,0.970788,0.999984,318.772436,"[[385324, 1, 0, 0, 0], [0, 330051, 1, 0, 0], [...","[0.9999987023923992, 0.9999969701743968, 0.984..."
2,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,KNN,0.999955,0.955743,0.910117,0.929933,0.999954,0.999955,0.999954,0.910117,0.999914,43.471231,"[[385320, 3, 0, 2, 0], [1, 330050, 1, 0, 0], [...","[0.9999844287694056, 0.9999909105507298, 0.936..."
3,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,LinearSVM,0.869625,0.604025,0.670217,0.610370,0.881580,0.869625,0.869633,0.670217,0.761230,3234.900379,"[[309118, 75687, 0, 520, 0], [15498, 314547, 0...","[0.8667726585742687, 0.8733741217237746, 0.446..."
4,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,LogisticRegression,0.899679,0.541709,0.862882,0.549766,0.902305,0.899679,0.900773,0.862882,0.808301,1591.392881,"[[344967, 37435, 15, 2907, 1], [28646, 301223,...","[0.9062898592226442, 0.900844402376349, 0.1261..."
5,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,LDA,0.737253,0.705582,0.507692,0.540688,0.765263,0.737253,0.726634,0.507692,0.501678,2.865868,"[[348382, 36943, 0, 0, 0], [141627, 188425, 0,...","[0.7851065436062005, 0.6784222654280982, 0.875..."
6,Exp1_Top10_CombinedRank,FeatureSelection,Top10_MI_ChiSq_Fisher,10,GaussianNB,0.471424,0.434221,0.521816,0.370458,0.500937,0.471424,0.309846,0.521816,0.189527,1.184056,"[[1499, 383785, 0, 41, 0], [431, 329619, 0, 2,...","[0.0077217889412031365, 0.630480522411755, 0.3..."


In [28]:
output_path = "/kaggle/working/final_results_all_experiments.csv"

if os.path.exists(output_path):
    results_df.to_csv(output_path, mode="a", header=False, index=False)
else:
    results_df.to_csv(output_path, index=False)


In [16]:
RESULTS_PATH = "/kaggle/working/final_results_all_experiments.csv"
import pandas as pd
import os

results_df = pd.DataFrame(results)
def append_results_to_csv(results_df, path):
    if os.path.exists(path):
        results_df.to_csv(
            path,
            mode="a",
            header=False,
            index=False
        )
    else:
        results_df.to_csv(
            path,
            index=False
        )

append_results_to_csv(results_df, RESULTS_PATH)


In [30]:
results.clear()


In [31]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp2_Top5_CombinedRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "Top5_MI_ChiSq_Fisher",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)


In [32]:
top_5_features = rank_df['feature'].head(5).tolist()
print("Top 5 selected features:", top_5_features)

Top 5 selected features: ['AR_P_Proto_P_DstIP', 'AR_P_Proto_P_SrcIP', 'TnBPSrcIP', 'dur', 'TnBPDstIP']


In [33]:
X_train_sel = X_train[top_5_features].copy()
X_test_sel  = X_test[top_5_features].copy()

In [34]:
# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

In [35]:
results = []
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)


✅ GaussianNB finished in 0.81s
✅ LDA finished in 1.74s
✅ LogisticRegression finished in 854.12s
✅ LinearSVM finished in 96.44s
✅ KNN finished in 23.35s
✅ DecisionTree finished in 23.21s
✅ RandomForest finished in 245.05s


In [36]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()


In [37]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp3_Top15_CombinedRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "Top15_MI_ChiSq_Fisher",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)


In [38]:
top_15_features = rank_df['feature'].head(15).tolist()
print("Top 15 selected features:", top_15_features)

X_train_sel = X_train[top_15_features].copy()
X_test_sel  = X_test[top_15_features].copy()

# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

Top 15 selected features: ['AR_P_Proto_P_DstIP', 'AR_P_Proto_P_SrcIP', 'TnBPSrcIP', 'dur', 'TnBPDstIP', 'TnP_PerProto', 'AR_P_Proto_P_Dport', 'AR_P_Proto_P_Sport', 'sbytes', 'rate', 'bytes', 'N_IN_Conn_P_DstIP', 'dport', 'sum', 'max']


In [39]:
results = []
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)


✅ GaussianNB finished in 1.52s
✅ LDA finished in 6.18s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ LogisticRegression finished in 1762.04s


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


✅ LinearSVM finished in 8557.97s
✅ KNN finished in 159.91s
✅ DecisionTree finished in 67.60s
✅ RandomForest finished in 326.80s


In [40]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()


In [41]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp4_Top20_CombinedRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "Top20_MI_ChiSq_Fisher",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)


In [42]:
top_20_features = rank_df['feature'].head(20).tolist()
print("Top 20 selected features:", top_20_features)

X_train_sel = X_train[top_20_features].copy()
X_test_sel  = X_test[top_20_features].copy()

# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

Top 20 selected features: ['AR_P_Proto_P_DstIP', 'AR_P_Proto_P_SrcIP', 'TnBPSrcIP', 'dur', 'TnBPDstIP', 'TnP_PerProto', 'AR_P_Proto_P_Dport', 'AR_P_Proto_P_Sport', 'sbytes', 'rate', 'bytes', 'N_IN_Conn_P_DstIP', 'dport', 'sum', 'max', 'mean', 'TnP_Per_Dport', 'TnP_PSrcIP', 'TnP_PDstIP', 'srate']


In [43]:
results=[]
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)



✅ GaussianNB finished in 1.94s
✅ LDA finished in 6.50s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ LogisticRegression finished in 1901.71s


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


✅ LinearSVM finished in 8450.79s
✅ KNN finished in 5962.05s
✅ DecisionTree finished in 98.97s
✅ RandomForest finished in 423.79s


In [44]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()


In [20]:

# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [21]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp5_all_CombinedRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "all_MI_ChiSq_Fisher",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)


In [22]:
results= []
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)


✅ GaussianNB finished in 2.58s
✅ LDA finished in 10.51s
✅ LogisticRegression finished in 1782.23s


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


✅ LinearSVM finished in 18772.92s
✅ KNN finished in 5555.68s
✅ DecisionTree finished in 154.56s
✅ RandomForest finished in 334.81s


In [23]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()


In [24]:
top_15_features = mi_df['feature'].head(15).tolist()
print("Top 15 selected features:", top_15_features)

X_train_sel = X_train[top_15_features].copy()
X_test_sel  = X_test[top_15_features].copy()

# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

Top 15 selected features: ['dur', 'AR_P_Proto_P_DstIP', 'AR_P_Proto_P_SrcIP', 'srate', 'rate', 'AR_P_Proto_P_Sport', 'AR_P_Proto_P_Dport', 'sum', 'mean', 'max', 'stddev', 'TnBPDstIP', 'N_IN_Conn_P_DstIP', 'min', 'TnBPSrcIP']


In [25]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp6_15_midfRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "top15_MI_only",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)


In [26]:
results= []
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)


✅ GaussianNB finished in 0.92s
✅ LDA finished in 2.64s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ LogisticRegression finished in 1163.62s
✅ LinearSVM finished in 1067.82s
✅ KNN finished in 75.13s
✅ DecisionTree finished in 75.07s
✅ RandomForest finished in 321.11s


In [27]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()


In [18]:
top_15_features = chi_df['feature'].head(15).tolist()
print("Top 15 selected features:", top_15_features)

X_train_sel = X_train[top_15_features].copy()
X_test_sel  = X_test[top_15_features].copy()

# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

Top 15 selected features: ['bytes', 'dbytes', 'sbytes', 'TnBPSrcIP', 'TnBPDstIP', 'AR_P_Proto_P_Dport', 'AR_P_Proto_P_Sport', 'AR_P_Proto_P_SrcIP', 'AR_P_Proto_P_DstIP', 'rate', 'dport', 'seq', 'srate', 'TnP_PerProto', 'dpkts']


In [19]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp7_15_chi_dfRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "top15_chI_only",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)


In [20]:
results= []
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)



✅ GaussianNB finished in 1.49s
✅ LDA finished in 3.99s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ LogisticRegression finished in 2027.68s


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


✅ LinearSVM finished in 14571.21s
✅ KNN finished in 119.50s
✅ DecisionTree finished in 54.56s
✅ RandomForest finished in 279.05s


In [21]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()


In [17]:
top_15_features = fisher_df['feature'].head(15).tolist()
print("Top 15 selected features:", top_15_features)

X_train_sel = X_train[top_15_features].copy()
X_test_sel  = X_test[top_15_features].copy()

# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)

Top 15 selected features: ['TnP_PerProto', 'state_CON', 'dport', 'N_IN_Conn_P_DstIP', 'AR_P_Proto_P_DstIP', 'dur', 'proto_icmp', 'state_URP', 'mean', 'max', 'TnBPSrcIP', 'state_FIN', 'TnP_PSrcIP', 'sum', 'TnP_Per_Dport']


In [18]:
EXPERIMENT_CONFIG = {
    "experiment_id": "Exp8_15_fisher_dfRank",
    "experiment_type": "FeatureSelection",
    "feature_set": "top15_fisher_only",
    "num_features": X_train_scaled.shape[1]
}

# Train all models again (same cell)
# train_evaluate(...)



In [19]:
results= []
# 1️⃣ Gaussian Naive Bayes (very fast)
train_evaluate(
    GaussianNB(),
    "GaussianNB",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 2️⃣ LDA (very fast)
train_evaluate(
    LinearDiscriminantAnalysis(),
    "LDA",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 3️⃣ Logistic Regression
train_evaluate(
    LogisticRegression(max_iter=2000),
    "LogisticRegression",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)
# 4️⃣ Linear SVM
train_evaluate(
    LinearSVC(max_iter=3000),
    "LinearSVM",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)

# 5️⃣ KNN
train_evaluate(
    KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "KNN",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)

# 6️⃣ Decision Tree
train_evaluate(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    sample_weight=sample_weights,
    experiment_config=EXPERIMENT_CONFIG
)
# 7️⃣ Random Forest (reduced size, parallel)
train_evaluate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest",
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    experiment_config=EXPERIMENT_CONFIG
)


✅ GaussianNB finished in 1.48s
✅ LDA finished in 4.27s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ LogisticRegression finished in 1593.73s
✅ LinearSVM finished in 3333.12s
✅ KNN finished in 1091.08s
✅ DecisionTree finished in 42.64s
✅ RandomForest finished in 334.37s


In [20]:
results_df = pd.DataFrame(results)
append_results_to_csv(results_df, RESULTS_PATH)
results.clear()
